# 05. 메모리 대화 추적

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `ConversationBufferMemory`를 사용한 멀티턴 대화가 LangSmith에서 어떻게 기록되는지 설명할 수 있어요
2. `thread_id`를 사용해 여러 대화 턴을 하나의 세션으로 그룹핑할 수 있어요
3. 대화가 누적됨에 따라 입력 토큰이 증가하는 패턴을 LangSmith에서 관찰할 수 있어요
4. LangSmith Threads 뷰에서 대화 히스토리를 탐색할 수 있어요

## 사전 지식

- `04-Chain-and-LCEL-Tracing` 노트북 완료
- `ConversationBufferMemory` 기본 사용법
- `ChatPromptTemplate`, LCEL 체인 구성 경험

## 개념 설명: 멀티턴 대화와 Thread

챗봇 대화는 여러 턴으로 이루어져요. LangSmith에서 각 턴은 별도의 Trace로 기록되지만, `thread_id`를 설정하면 같은 대화에 속한 Trace들을 하나의 **Thread**로 묶어서 볼 수 있어요.

### 멀티턴 Trace 구조

```mermaid
flowchart TD
    subgraph Thread ["thread_id: abc-123"]
        T1["턴 1 Trace<br/>'안녕하세요, 저는 Bob이에요'"]
        T2["턴 2 Trace<br/>'제 이름이 뭔가요?'"]
        T3["턴 3 Trace<br/>'오늘 날씨는 어때요?'"]
    end

    T1 --> T2 --> T3

    T1 --> TK1["토큰: 입력 20, 출력 15"]
    T2 --> TK2["토큰: 입력 45, 출력 12"]
    T3 --> TK3["토큰: 입력 78, 출력 20"]

    classDef trace fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class T1,T2,T3 trace
    class TK1,TK2,TK3 storage
```

### Thread ID 설정 방법

| 방법 | 코드 | 설명 |
|------|------|------|
| `tracing_context` | `with ls.tracing_context(metadata={"thread_id": tid})` | 코드 블록 내 모든 Trace에 적용 |
| `config` | `chain.invoke(..., config={"metadata": {"thread_id": tid}})` | 단일 호출에만 적용 |

> 🔑 **핵심 개념**: `thread_id` 외에도 `session_id`, `conversation_id` 키를 사용할 수 있어요. LangSmith는 이 세 키를 모두 Thread 그룹핑에 사용해요.

### UUID v7 사용 권장

Thread ID는 시간 순서를 보장하는 **UUID v7**을 사용하면 LangSmith에서 시간순으로 정렬이 쉬워요.

## 환경 설정

In [1]:
# ---------------------------------------------------
# 환경변수 로드 및 프로젝트 설정
# ---------------------------------------------------
from dotenv import load_dotenv
import os

load_dotenv()

# 이 노트북 전용 프로젝트
os.environ["LANGSMITH_PROJECT"] = "Ch05-Memory-Tracing"

print(f"프로젝트: {os.environ['LANGSMITH_PROJECT']}")

프로젝트: Ch05-Memory-Tracing


## 1. ConversationBufferMemory 기본 설정

`ConversationBufferMemory`는 대화 기록을 메모리에 저장해요. 각 대화 턴마다 이전 메시지가 누적되어 LLM에 전달돼요.

In [2]:
# ---------------------------------------------------
# ConversationBufferMemory와 LCEL 체인 구성
# ---------------------------------------------------
# 대화 기록을 유지하는 챗봇 체인을 만들어요
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.memory import ConversationBufferMemory
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough

# 기본 모델
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 대화 기록을 포함하는 프롬프트
# MessagesPlaceholder는 이전 대화 메시지가 들어갈 자리예요
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트예요. 사용자의 이름을 기억해 대화해요."),
    MessagesPlaceholder(variable_name="history"),  # 대화 히스토리가 여기 들어가요
    ("human", "{input}")
])

# 메모리 초기화: return_messages=True로 메시지 형식으로 반환해요
memory = ConversationBufferMemory(
    memory_key="history",  # 프롬프트의 변수명과 일치해야 해요
    return_messages=True   # LangChain 메시지 형식으로 반환
)

# 메모리를 포함한 체인 구성
def chain_with_memory(input_text: str) -> str:
    """메모리를 사용하는 체인 실행 함수예요"""
    # 1. 현재 메모리에서 대화 기록 로드
    history = memory.chat_memory.messages

    # 2. 프롬프트 → 모델 → 파서 체인 실행
    chain = chat_prompt | model | StrOutputParser()
    response = chain.invoke({
        "input": input_text,
        "history": history
    })

    # 3. 메모리에 대화 저장
    memory.save_context(
        {"input": input_text},
        {"output": response}
    )

    return response

print("메모리 챗봇 체인 준비 완료!")

메모리 챗봇 체인 준비 완료!


/var/folders/k2/q6_f_rp50lqg7l24d_3dk1vh0000gn/T/ipykernel_4603/926916298.py:23: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


## 2. thread_id로 멀티턴 대화 추적

`thread_id`를 사용하면 여러 대화 턴을 하나의 세션으로 묶을 수 있어요. LangSmith의 **Threads 뷰**에서 전체 대화를 순서대로 볼 수 있어요.

> 🎯 **강의 포인트**: 대화가 진행될수록 각 Trace의 입력 토큰이 증가하는 것을 LangSmith에서 확인해 보세요. 이것이 `ConversationBufferMemory`의 핵심 특성이에요.

In [3]:
# ---------------------------------------------------
# UUID v7로 thread_id 생성
# ---------------------------------------------------
# LangSmith 권장 방식: UUID v7 사용 (시간 순서 보장)
import uuid  # langsmith 0.3.x에서는 uuid7이 없으므로 uuid4 사용

# 이 대화 세션의 고유 ID를 생성해요
thread_id = str(uuid.uuid4())

print(f"생성된 Thread ID: {thread_id}")
print("이 ID로 LangSmith에서 대화 전체를 추적할 수 있어요")

생성된 Thread ID: 642d1c92-9e45-471b-80a6-ed432cf1b7a3
이 ID로 LangSmith에서 대화 전체를 추적할 수 있어요


In [4]:
# ---------------------------------------------------
# 멀티턴 대화 실행: tracing_context로 thread_id 설정
# ---------------------------------------------------
# 각 대화 턴에 동일한 thread_id를 설정해요
import langsmith as ls

# 대화 턴 1: 자기 소개
print("=== 대화 턴 1 ===")
with ls.tracing_context(metadata={"thread_id": thread_id, "turn": 1}):
    response1 = chain_with_memory("안녕하세요! 저는 Alice예요.")
print(f"AI: {response1}")

# 대화 턴 2: 이름 확인
print("\n=== 대화 턴 2 ===")
with ls.tracing_context(metadata={"thread_id": thread_id, "turn": 2}):
    response2 = chain_with_memory("제 이름이 뭔지 기억하나요?")
print(f"AI: {response2}")

# 대화 턴 3: 새로운 주제
print("\n=== 대화 턴 3 ===")
with ls.tracing_context(metadata={"thread_id": thread_id, "turn": 3}):
    response3 = chain_with_memory("파이썬 프로그래밍 배우는 게 어렵나요?")
print(f"AI: {response3}")

=== 대화 턴 1 ===
AI: 안녕하세요, Alice! 만나서 반가워요. 오늘은 어떤 이야기를 나눠볼까요?

=== 대화 턴 2 ===
AI: 네, Alice님! 당신의 이름을 기억하고 있어요. 다른 궁금한 점이나 이야기하고 싶은 주제가 있나요?

=== 대화 턴 3 ===
AI: 파이썬은 비교적 배우기 쉬운 프로그래밍 언어로 알려져 있어요. 문법이 간단하고 읽기 쉬워서 초보자에게 적합하죠. 기본적인 개념을 이해하고 나면, 다양한 프로젝트를 통해 실력을 키울 수 있어요. 처음에는 조금 어려울 수 있지만, 꾸준히 연습하면 점점 더 익숙해질 거예요. 어떤 부분이 특히 궁금한가요?


### LangSmith Threads 뷰 확인 방법

1. 프로젝트 페이지에서 좌측 메뉴 **Threads** 클릭
2. 생성된 Thread ID로 검색
3. 3개의 턴이 시간순으로 정렬된 것을 확인
4. 각 턴 클릭 시 해당 Trace 상세 정보 표시

## 3. 토큰 증가 패턴 관찰

`ConversationBufferMemory`는 모든 이전 대화를 누적해서 전달해요. 대화가 길어질수록 입력 토큰이 선형적으로 증가해요.

> ⚠️ **자주 하는 실수**: `ConversationBufferMemory`를 장기간 사용하면 컨텍스트 윈도우 한계(예: gpt-4o-mini는 128K 토큰)에 도달할 수 있어요. LangSmith에서 토큰 증가 추세를 모니터링하는 것이 중요해요.

In [5]:
# ---------------------------------------------------
# 토큰 증가 패턴 관찰
# ---------------------------------------------------
# 새로운 대화 세션을 시작해서 토큰 증가를 관찰해요

# 새 메모리와 thread_id로 새 세션 시작
memory2 = ConversationBufferMemory(
    memory_key="history",
    return_messages=True
)
thread_id2 = str(uuid.uuid4())

def chat_and_measure(input_text: str, memory_obj, tid: str) -> dict:
    """대화하면서 토큰 수를 측정해요"""
    history = memory_obj.chat_memory.messages
    chain = chat_prompt | model | StrOutputParser()

    # 응답과 메타데이터를 함께 가져와요
    ai_message = model.invoke(
        chat_prompt.format_messages(input=input_text, history=history)
    )

    response_text = ai_message.content
    token_usage = ai_message.response_metadata.get("token_usage", {})

    # 메모리에 저장
    memory_obj.save_context(
        {"input": input_text},
        {"output": response_text}
    )

    return {
        "response": response_text,
        "input_tokens": token_usage.get("prompt_tokens", 0),
        "output_tokens": token_usage.get("completion_tokens", 0)
    }

# 5턴 대화 진행하며 토큰 증가 관찰
conversations = [
    "안녕하세요! 저는 Bob이에요.",
    "LangChain이 뭔지 설명해 줄 수 있나요?",
    "그럼 LangSmith는요?",
    "제 이름이 뭔지 기억하나요?",
    "오늘 배운 내용을 정리해 줄 수 있나요?"
]

print("턴  | 입력 | 출력 | 질문")
print("-" * 60)

for i, question in enumerate(conversations, 1):
    with ls.tracing_context(metadata={"thread_id": thread_id2, "turn": i}):
        result = chat_and_measure(question, memory2, thread_id2)
    print(f"턴 {i} | {result['input_tokens']:4d} | {result['output_tokens']:4d} | {question[:30]}...")

print("\n입력 토큰이 대화가 누적될수록 증가하는 것을 확인하세요!")

턴  | 입력 | 출력 | 질문
------------------------------------------------------------
턴 1 |   45 |   23 | 안녕하세요! 저는 Bob이에요....
턴 2 |   89 |  301 | LangChain이 뭔지 설명해 줄 수 있나요?...
턴 3 |  405 |  279 | 그럼 LangSmith는요?...
턴 4 |  702 |   30 | 제 이름이 뭔지 기억하나요?...
턴 5 |  752 |  300 | 오늘 배운 내용을 정리해 줄 수 있나요?...

입력 토큰이 대화가 누적될수록 증가하는 것을 확인하세요!


## 4. config로 thread_id 설정 (단일 호출)

`tracing_context` 대신 체인 호출 시 `config` 파라미터로 `thread_id`를 직접 설정할 수도 있어요.

> 💡 **실무 팁**: 웹 서버나 API에서 사용자 세션 ID를 `thread_id`로 사용하면 사용자별 대화 히스토리를 LangSmith에서 쉽게 추적할 수 있어요.

In [6]:
# ---------------------------------------------------
# config로 thread_id 설정하는 방법
# ---------------------------------------------------
# LCEL 체인의 invoke()에 config를 직접 전달해요

# 새 메모리 초기화
memory3 = ConversationBufferMemory(
    memory_key="history",
    return_messages=True
)
thread_id3 = str(uuid.uuid4())

# LCEL 체인
simple_chain = chat_prompt | model | StrOutputParser()

# 대화 턴 1
response = simple_chain.invoke(
    {
        "input": "안녕하세요! 저는 Charlie예요.",
        "history": memory3.chat_memory.messages
    },
    config={
        "run_name": "chat-turn-1",        # Trace 이름
        "metadata": {
            "thread_id": thread_id3,       # 대화 세션 ID
            "user_name": "Charlie"         # 추가 메타데이터
        }
    }
)
memory3.save_context({"input": "안녕하세요! 저는 Charlie예요."}, {"output": response})
print(f"턴 1 응답: {response}")

# 대화 턴 2
response2 = simple_chain.invoke(
    {
        "input": "제 이름이 뭔지 알 수 있나요?",
        "history": memory3.chat_memory.messages
    },
    config={
        "run_name": "chat-turn-2",
        "metadata": {
            "thread_id": thread_id3,
            "user_name": "Charlie"
        }
    }
)
print(f"\n턴 2 응답: {response2}")
print(f"\nThread ID: {thread_id3}")
print("LangSmith Threads 뷰에서 'Charlie' 대화를 찾아보세요!")

턴 1 응답: 안녕하세요, Charlie! 만나서 반가워요. 오늘은 어떤 이야기를 나눠볼까요?

턴 2 응답: 네, Charlie라고 말씀하셨죠! 당신의 이름을 기억하고 있어요. 다른 질문이나 이야기하고 싶은 주제가 있나요?

Thread ID: a55fbb30-9567-45fb-bb27-004249b15030
LangSmith Threads 뷰에서 'Charlie' 대화를 찾아보세요!


## 5. 실습: 나만의 대화 추적 실험

In [7]:
# ============================================================
# TODO: 아래 대화 목록을 수정해서 직접 대화를 진행하고
#       LangSmith Threads 뷰에서 대화 히스토리를 탐색해 보세요
# 힌트: 대화 내용을 길게 만들수록 토큰 증가 패턴이 더 명확하게 나타나요
# 예상 결과: 각 턴마다 입력 토큰이 약 (이전 응답 토큰 수)만큼 증가해요
# ============================================================

from langchain_core.tracers.langchain import wait_for_all_tracers

# 나만의 대화 내용으로 바꿔 보세요!
MY_CONVERSATIONS = [
    "안녕! 나는 학생이야.",
    "LangSmith가 왜 유용한지 알려줘.",
    "내가 뭐라고 소개했어?"
]

# 새 세션 설정
my_memory = ConversationBufferMemory(memory_key="history", return_messages=True)
my_thread_id = str(uuid.uuid4())
print(f"내 Thread ID: {my_thread_id}")

# 대화 진행
for i, question in enumerate(MY_CONVERSATIONS, 1):
    history = my_memory.chat_memory.messages
    chain = chat_prompt | model | StrOutputParser()

    with ls.tracing_context(metadata={"thread_id": my_thread_id, "turn": i}):
        answer = chain.invoke({"input": question, "history": history})

    my_memory.save_context({"input": question}, {"output": answer})

    print(f"\n[턴 {i}] 나: {question}")
    print(f"       AI: {answer[:200]}")

# 모든 Trace 전송 완료
wait_for_all_tracers()
print(f"\n모든 대화 Trace 기록 완료!")
print(f"LangSmith Threads 뷰에서 thread_id = {my_thread_id[:8]}... 를 검색해 보세요!")

내 Thread ID: 657240c8-3eb5-419e-80ee-b4960449c006

[턴 1] 나: 안녕! 나는 학생이야.
       AI: 안녕! 만나서 반가워. 어떤 과목을 공부하고 있어?

[턴 2] 나: LangSmith가 왜 유용한지 알려줘.
       AI: LangSmith는 언어 학습과 관련된 다양한 기능을 제공하는 플랫폼으로, 여러 가지 이유로 유용할 수 있어요. 예를 들어:

1. **다양한 언어 지원**: 여러 언어를 지원하여 다양한 언어를 배우고 연습할 수 있어요.
2. **인터랙티브 학습**: 게임이나 퀴즈 형식으로 학습할 수 있어 지루하지 않게 공부할 수 있어요.
3. **개인 맞춤형 학습**: 

[턴 3] 나: 내가 뭐라고 소개했어?
       AI: 너는 "나는 학생이야"라고 소개했어. 어떤 과목을 공부하고 있는지 궁금해!

모든 대화 Trace 기록 완료!
LangSmith Threads 뷰에서 thread_id = 657240c8... 를 검색해 보세요!


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **멀티턴 Trace**: 각 대화 턴은 별도의 Trace로 기록되지만 `thread_id`로 연결돼요
- **Thread ID 키**: `thread_id`, `session_id`, `conversation_id` 중 하나를 metadata에 설정하면 돼요
- **UUID v7**: `uuid.uuid4()`로 고유한 Thread ID를 생성해요 (langsmith 최신 버전에서는 `uuid7` 사용 가능)
- **토큰 증가 패턴**: `ConversationBufferMemory`는 대화가 누적될수록 입력 토큰이 선형 증가해요
- **tracing_context vs config**: `tracing_context`는 여러 호출을 묶을 때, `config`는 단일 호출에 적용해요

## 다음 노트북 예고

다음 `06-Metadata-Tags-Filtering.ipynb`에서는 **메타데이터와 태그를 활용한 고급 필터링**을 배워요. LangSmith UI에서 수백 개의 Trace 중 원하는 것을 빠르게 찾는 방법과 dev/staging/prod 환경별 프로젝트 분리 전략을 살펴볼게요.